In [1]:
# import sys
# !{sys.executable} -m pip uninstall prettytable -y
# !{sys.executable} -m pip uninstall ipython-sql -y
# !{sys.executable} -m pip install prettytable==3.9.0
# !{sys.executable} -m pip install ipython-sql==0.4.1

In [2]:
# !pip uninstall sqlalchemy -y
# !pip install sqlalchemy==1.4.49

In [3]:
%load_ext sql

In [4]:
%config SqlMagic.autopandas = True
%config SqlMagic.displaycon = False

In [5]:
import sys
print(sys.executable)

f:\Python\FastAPI01\myenv\Scripts\python.exe


In [6]:
import os

In [7]:
host = os.getenv('SQL_HOST', 'localhost')
port = os.getenv('SQL_PORT', '5432')
user = os.getenv('SQL_USER')
password = os.getenv('SQL_PASSWORD')
database = os.getenv('SQL_DB', 'sdb')

In [8]:
connection_string = f"postgresql://{user}:{password}@{host}:{port}/{database}"

In [9]:
from sqlalchemy import create_engine

engine = create_engine(connection_string)

In [10]:

%sql $connection_string

In [11]:
%%sql
SELECT * FROM nyc_neighborhoods WHERE FALSE

0 rows affected.


""


In [12]:
%%sql 
SELECT * FROM nyc_neighborhoods LIMIT 5;

5 rows affected.


,id,geom,boroname,name
0,1,0106000020266900000100000001030000000100000011...,Brooklyn,Bensonhurst
1,2,0106000020266900000100000001030000000100000008...,Manhattan,East Village
2,3,0106000020266900000100000001030000000100000034...,Manhattan,West Village
3,4,0106000020266900000100000001030000000100000057...,The Bronx,Throggs Neck
4,5,0106000020266900000100000001030000000100000066...,The Bronx,Wakefield-Williamsbridge


In [13]:
%%sql
SELECT * FROM nyc_neighborhoods WHERE boroname LIKE '%Manhattan%';

28 rows affected.


,id,geom,boroname,name
0,2,0106000020266900000100000001030000000100000008...,Manhattan,East Village
1,3,0106000020266900000100000001030000000100000034...,Manhattan,West Village
2,7,010600002026690000010000000103000000010000001F...,Manhattan,Battery Park
3,8,0106000020266900000100000001030000000100000006...,Manhattan,Carnegie Hill
4,11,0106000020266900000100000001030000000100000017...,Manhattan,Harlem
5,12,0106000020266900000100000001030000000100000016...,Manhattan,Gramercy
6,22,0106000020266900000100000001030000000100000012...,Manhattan,Soho
7,30,010600002026690000010000000103000000010000000C...,Manhattan,Morningside Heights
8,31,0106000020266900000100000001030000000100000007...,Manhattan,Murray Hill
9,48,010600002026690000010000000103000000010000000D...,Manhattan,Central Park


In [14]:
%%sql
SELECT avg(char_length(name)), stddev(char_length(name)) 
FROM nyc_neighborhoods
WHERE boroname = 'Manhattan';

1 rows affected.


,avg,stddev
0,11.8214285714285714,4.3123729948325257


In [15]:
%%sql
SELECT boroname, avg(char_length(name)), stddev(char_length(name)) 
FROM nyc_neighborhoods
GROUP BY boroname;

5 rows affected.


,boroname,avg,stddev
0,Queens,11.6666666666666667,5.0057438272815975
1,Brooklyn,11.7391304347826087,3.9105613559407395
2,Staten Island,12.2916666666666667,5.2043390480959474
3,The Bronx,12.0416666666666667,3.6651017740975152
4,Manhattan,11.8214285714285714,4.3123729948325257


In [16]:
%%sql
SELECT * FROM nyc_census_blocks LIMIT 5;

5 rows affected.


,id,geom,blkid,popn_total,popn_white,popn_black,popn_nativ,popn_asian,popn_other,boroname
0,1,0106000000010000000103000000010000000A00000051...,360850009001000,97,51,32,1,5,8,Staten Island
1,2,0106000000010000000103000000010000000700000008...,360850020011000,66,52,2,0,7,5,Staten Island
2,3,0106000000010000000103000000010000000600000082...,360850040001000,62,14,18,2,25,3,Staten Island
3,4,0106000000010000000103000000010000000A00000011...,360850074001000,137,92,12,0,13,20,Staten Island
4,5,01060000000100000001030000000100000014000000EA...,360850096011000,289,230,0,0,32,27,Staten Island


In [17]:
%%sql
SELECT Sum(popn_total) AS total_population
FROM nyc_census_blocks
WHERE boroname = 'Manhattan';

1 rows affected.


,total_population
0,1585873


In [18]:
%%sql
SELECT DISTINCT boroname, Sum(popn_total) AS total_population
FROM nyc_census_blocks
GROUP BY boroname
ORDER BY total_population DESC;

5 rows affected.


,boroname,total_population
0,Brooklyn,2504700
1,Queens,2230621
2,Manhattan,1585873
3,The Bronx,1385108
4,Staten Island,468730


In [22]:
%%sql
SELECT boroname,
100*Sum(popn_white)/Sum(popn_total) AS pct_white,
100*Sum(popn_black)/Sum(popn_total) AS pct_black,
100*Sum(popn_asian)/Sum(popn_total) AS pct_asian,
100*Sum(popn_other)/Sum(popn_total) AS pct_other
FROM nyc_census_blocks
GROUP BY boroname;

5 rows affected.


,boroname,pct_white,pct_black,pct_asian,pct_other
0,Queens,39.7220773945910130,19.1253467083830019,22.9428486506672357,17.5144500119025150
1,Brooklyn,42.8011737932686549,34.3387631253243901,10.4713538547530642,11.8487643230726235
2,The Bronx,27.9037446899447552,36.4736901382419277,3.5815979692558270,30.7226584497382154
3,Manhattan,57.4493039480462811,15.5552809083703424,11.3219658825139214,15.1268102805205713
4,Staten Island,72.8942034860154033,10.6366138288566979,7.5019734175324814,8.6055938386704499
